# DEKM 2021 20NEWS official-source diagnostic, seed 42

This bounded run corrects the 20NEWS provenance mismatch found in the three-dataset matrix. The official DEKM `utils.py` ignores the Google Drive NPZ files for 20NEWS and instead executes `fetch_20newsgroups(subset="all")` followed by `TfidfVectorizer(max_features=2000)`, yielding the paper count of 18,846 rows. Only seed 42 is run first; seeds 43-44 are justified only if this source-faithful control moves materially toward Table III.


In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow:', tf.__version__)
print('GPUs:', gpus)
assert gpus, 'Enable a GPU accelerator in Kaggle notebook settings.'


In [ ]:
from pathlib import Path
import subprocess
import sys

REPO = Path('/kaggle/working/mvdec_fpmax')
subprocess.run(['rm', '-rf', str(REPO)], check=True)
subprocess.run(['git', 'clone', '--branch', 'dev', '--single-branch', 'https://github.com/vdat-18/mvdec_fpmax.git', str(REPO)], check=True)
EXPECTED_REVISION = '9af1f790c0d57e0ded31cbc5f779b986f8b1822f'
subprocess.run(['git', 'checkout', '--detach', EXPECTED_REVISION], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
if str(REPO / 'src') not in sys.path:
    sys.path.insert(0, str(REPO / 'src'))
actual = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
assert actual == EXPECTED_REVISION, actual
print('Repository revision:', actual)


In [ ]:
import json
import sklearn
from sklearn.datasets import fetch_20newsgroups

raw = fetch_20newsgroups(subset='all', download_if_missing=True)
assert len(raw.data) == 18846, len(raw.data)
class_counts = {str(label): int((raw.target == label).sum()) for label in range(20)}
source_audit = {'source_id': 'sklearn_fetch_20newsgroups_all_tfidf2000_18846', 'raw_rows': len(raw.data), 'class_counts': class_counts, 'sklearn_version': sklearn.__version__}
OUTPUT = Path('/kaggle/working/dekm-2021-20news-sklearn18846-seed42')
RUN_ROOT = OUTPUT / 'runs'
OUTPUT.mkdir(parents=True, exist_ok=True)
(OUTPUT / 'source_audit.json').write_text(json.dumps(source_audit, indent=2), encoding='utf-8')
print(json.dumps(source_audit, indent=2))
command = [
    sys.executable, '-m', 'representation_learning.DEKM_dense', '20NEWS',
    '--protocol', 'release-2021-sklearn-20news', '--seed', '42', '--runs', '1',
    '--require-gpu', '--dataset-root', str(REPO / 'external_repos/DEKM/datasets'),
    '--output-dir', str(RUN_ROOT),
]
subprocess.run(command, cwd=REPO, check=True)


In [ ]:
import hashlib
import pandas as pd
from pipeline.external_metrics import compute_external_metrics

PROTOCOL = 'dekm_2021_20news_sklearn18846_seeded_modern_runtime_v1'
protocol_root = RUN_ROOT / '20news' / PROTOCOL
config_roots = [path for path in protocol_root.iterdir() if path.is_dir()]
assert len(config_roots) == 1, config_roots
run_dir = config_roots[0] / 'seed_42'
manifest = json.loads((run_dir / 'manifest.json').read_text(encoding='utf-8'))
assert manifest['status'] == 'complete'
assert manifest['protocol_id'] == PROTOCOL
assert manifest['source_id'] == 'sklearn_fetch_20newsgroups_all_tfidf2000_18846'
assert manifest['n_samples'] == 18846
assert manifest['n_features'] == 2000
assert manifest['n_clusters'] == 20
assert manifest['config']['refinement_batch_size'] == 256
assert manifest['config']['assignment_change_tolerance'] == 0.005
assert manifest['config']['input_normalization'] == 'release_l2_unit_norm'
assert manifest['official_repository_paper_era_revision'] == '129b7d08b545f02d920ccff0948ec44b654b3158'
assert manifest['official_repository_paper_era_utils_sha256'] == '93b18e8b09559b2ddd74fe631e44670ead935e7dd962502e5aa0d0d2b22f4451'
assignments = pd.read_csv(run_dir / 'assignments.csv')
metrics = compute_external_metrics(assignments['true_label'].to_numpy(), assignments['cluster'].to_numpy(), n_clusters=20)
assert abs(metrics.acc - manifest['metrics']['acc']) < 1e-12
assert abs(metrics.nmi - manifest['metrics']['nmi']) < 1e-12
for name, filename in manifest['paths'].items():
    path = run_dir / filename
    assert hashlib.sha256(path.read_bytes()).hexdigest() == manifest['output_sha256'][name]
paper = {'acc': 0.4108, 'nmi': 0.4027}
release_npz = {'acc_mean': 0.3605250041535138, 'nmi_mean': 0.39883357051129753}
diagnostic = {'dataset': '20NEWS', 'seed': 42, 'protocol_id': PROTOCOL, 'source_id': manifest['source_id'], 'source_sha256': manifest['source_sha256'], 'n_samples': manifest['n_samples'], 'acc': float(metrics.acc), 'nmi': float(metrics.nmi), 'fit_time_seconds': float(manifest['fit_time_seconds']), 'steps_completed': int(manifest['steps_completed']), 'refreshes_completed': int(manifest['refreshes_completed']), 'stop_reason': manifest['stop_reason'], 'paper_acc': paper['acc'], 'paper_nmi': paper['nmi'], 'paper_acc_delta_pp': float(100 * (metrics.acc - paper['acc'])), 'paper_nmi_delta_pp': float(100 * (metrics.nmi - paper['nmi'])), 'npz_matrix_acc_mean': release_npz['acc_mean'], 'npz_matrix_nmi_mean': release_npz['nmi_mean']}
pd.DataFrame([diagnostic]).to_csv(OUTPUT / 'diagnostic.csv', index=False)
(OUTPUT / 'diagnostic.json').write_text(json.dumps(diagnostic, indent=2), encoding='utf-8')
(OUTPUT / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
assignments.to_csv(OUTPUT / 'assignments.csv', index=False)
display(pd.DataFrame([diagnostic]))


In [ ]:
import shutil

zip_path = Path(shutil.make_archive('/kaggle/working/dekm-2021-20news-sklearn18846-seed42', 'zip', '/kaggle/working', 'dekm-2021-20news-sklearn18846-seed42'))
digest = hashlib.sha256(zip_path.read_bytes()).hexdigest()
sha_path = zip_path.with_suffix('.zip.sha256')
sha_path.write_text(f'{digest}  {zip_path.name}\n', encoding='ascii')
print('ZIP:', zip_path)
print('SHA-256:', digest)
